In [ ]:
# !huggingface-cli login # [REDACTED_SECRET]
!pip install deep-translator
!pip install datasets


In [ ]:
import kagglehub
import shutil
import os
# Download latest version
path = kagglehub.dataset_download("thedevastator/nlp-mental-health-conversations")
os.makedirs("./data")
shutil.move(path, "./data")

In [ ]:
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


In [ ]:
def translate_text(text):
  translated = GoogleTranslator(source='ar', target='en').translate(text)
  return translated

def translate_text_ar(text):
  translated = GoogleTranslator(source='en', target='ar').translate(text)
  return translated

In [ ]:
df=pd.read_csv('/content/data/train.csv')

mask=df['Response'].isnull()
df=df[~mask]

def pre_prroc(text):
  cleaned_text=text.replace('\xa0','').replace('\n','').strip()
  cleaned_text = re.sub(r'\s+', ' ', text)
  return cleaned_text

df['Context']=df['Context'].apply(pre_prroc)
df['Response']=df['Response'].apply(pre_prroc)

def form_string(question, answer):
    s = f"<|startoftext|>question: {str(question).strip()} " \
        f"answer: {str(answer).strip()}<|endoftext|>"
    return s

data = df.apply(lambda row: form_string(row['Context'], row['Response']), axis=1)

data.to_csv('/content/data/final_data.csv',index=False)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "openai-community/gpt2-medium"#@param ['openai-community/gpt2-large','openai-community/gpt2-medium']
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                              bos_token='<|startoftext|>',
                                              eos_token='<|endoftext|>',
                                              unk_token='<|unknown|>',
                                              pad_token='<|pad|>'
                                             )
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
model

In [ ]:
# from peft import LoraConfig,PeftModel,LoraModel,get_peft_model,TaskType

# config = LoraConfig(task_type=TaskType.CAUSAL_LM,r=32, lora_alpha=64, target_modules = ["c_attn", "c_proj"], lora_dropout=0.05, bias="none")

# model = get_peft_model(model, config)
# model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/data/final_data.csv")
dataset = dataset['train'].train_test_split(test_size=0.2).shuffle()

In [ ]:
# dataset
# # dataset['train'][10]

In [ ]:
def data_tokenization(batch):
  tokenized_inputs = tokenizer(batch['0'], truncation=True)  # Tokenize the batch
  batch['input_ids'] = tokenized_inputs['input_ids']  # Extract input_ids as a list
  batch['attention_mask'] = tokenized_inputs['attention_mask']  # Extract attention_mask as a list
  return batch

In [ ]:
dataset=dataset.map(data_tokenization,remove_columns=['0'],batched=True)

In [ ]:
# sample=dataset['train'][50]['input_ids']
# tokenizer.decode(sample)


In [ ]:
torch.cuda.empty_cache()


In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)


training_args = TrainingArguments(
    output_dir="./test_model",
    num_train_epochs=20,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=6,
    fp16=True,
    save_strategy="epoch",
    greater_is_better=False,
    load_best_model_at_end=True,
    logging_steps=25,
    learning_rate=1e-5,
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    evaluation_strategy='epoch',
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint=True) #resume_from_checkpoint=True #9f37d851f705cb04f6dffaa8e390714d3a97f36e

In [ ]:
text="انا قلق من فكرة الزواج ما رأيك"
ar_text=translate_text(text)

In [ ]:
ar_text

In [ ]:
def infer(prompt):
    input = f"<|startoftext|>question: {prompt.strip()}.answer:"
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]
    input_ids = input_ids.to(model.device)
    attention_mask = attention_mask.to(model.device)
    output = model.generate(input_ids,
                            attention_mask=attention_mask,
                            max_new_tokens=180,
                            do_sample = True, top_k = 50, top_p = 0.85)

    output = tokenizer.decode(output[0], skip_special_tokens=True)
    return output


res=infer("My grandson's step-mother sends him to school with a pink Barbie backpack as a form of punishment.")

In [ ]:
res

In [ ]:
res2=translate_text_ar(res)
res2

In [ ]:
t="""
م :
عت يحيحصتلا ةطخلا د  يف ةيضرم جئاتن اوققحي مل نيذلا بﻼطلا معدل ةيساسأ ةوطخ ة
 ﻻا ناحتم انل نيبت ثيح تا  دعب إ  ددع نأ انل نيبت بﻼطلل ةيناحتمﻻا ةيلمعلا ءارج
 نيرثعتملا  لوﻷا فصلا يف وه 226  ةبسنب بلاط 31   لوﻷا فصلا نم %  و 68
 ةبسنب بلاط 19  يناثلا فصلا يف % و ةطخ لمع متو إ  كلذو ةيحيحصت تاءارج
 يوتسم نيسحتل بﻼطلا .
"""

